# <font color='lightgreen'>01 load data 🚀</font>

    Cargamos los datos de las diferentes fuentes para su posterior análisis y visualización. Este proceso es fundamental para asegurar que los datos estén disponibles y listos para ser utilizados en las siguientes etapas del proyecto.
    Cargamos los datos de las diferentes fuentes desde GitHub, los normalizamos y los guardamos localmente.
    Este notebook es **idempotente**: se puede ejecutar varias veces sin duplicar datos ni romper el pipeline.
---

### <font color='lightgreen'>Librerías</font>

In [68]:
import pandas as pd
from pathlib import Path
import logging

print("✅ Librerías importadas")
print(f"Pandas versión: {pd.__version__}")


✅ Librerías importadas
Pandas versión: 3.0.2


### <font color='lightgreen'>Configuración</font>


#### <font color='lightgreen'>Configuración de logging</font>

In [69]:
# Configurar logging para tener trazabilidad
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

#### <font color='lightgreen'>Configuración de rutas</font>
Usamos `pathlib` para rutas absolutas relativas al proyecto, evitando `..` que son frágiles.

In [70]:
# Obtener la raíz del proyecto (asumiendo que este notebook está en notebooks/)
PROJECT_ROOT = Path.cwd().parent

# Carpeta donde guardaremos los datos descargados
DATA_LOADED = PROJECT_ROOT / "data" / "loaded"

# Crear la carpeta si no existe
DATA_LOADED.mkdir(parents=True, exist_ok=True)

logger.info(f"📁 Datos se guardarán en: {DATA_LOADED}")

2026-04-15 03:36:38,932 - INFO - 📁 Datos se guardarán en: c:\Users\marely\OneDrive\Documentos\Bussiness Inteligence Project\S03-26-Equipo-47-Business-Intelligence-development\S03-26-Equipo-47-Business-Intelligence-development\data\loaded


#### <font color='lightgreen'>Configuración de fuentes</font>

Definimos la URL base y los metadatos de cada archivo: nombre, separador, columnas de fecha, columnas numéricas que necesitan limpieza.

In [71]:
BASE_URL = "https://raw.githubusercontent.com/No-Country-simulation/S03-26-Equipo-47-Business-Intelligence/development/data/raw"

# Lista de archivos a descargar con su configuración
# Formato: (nombre_archivo, nombre_interno, separador, columnas_fecha, columnas_numericas_con_comas)
ARCHIVOS_CONFIG = [
    ("compras_proveedor.csv", "compras", ";", ["fecha_emision", "fecha_pago"], []),
    ("consumo_energetico.csv", "consumo_energetico", ";", ["fecha"], ["kwh_consumidos", "m3_gas"]),
    ("encuestas_clima.csv", "encuestas", ",", ["fecha_encuesta"], []),
    ("eventos_rrhh.csv", "eventos_rrhh", ",", ["fecha"], []),
    ("personal_nomina.csv", "personal_nomina", ",", ["mes", "fecha_ingreso"], []),
    ("residuos_reciclaje.csv", "residuos", ",", ["fecha_semana"], []),
    ("ventas_transacciones.csv", "ventas", ",", ["fecha"], []),
]

### <font color='lightgreen'>Función genérica de descarga y limpieza</font>

Esta función:
1. Descarga el CSV desde GitHub.
2. Convierte columnas de fecha a `datetime` (valores inválidos se convierten a `NaT`).
3. Para columnas numéricas, reemplaza comas por puntos y convierte a número.
4. Guarda el resultado en `data/loaded/`.
5. Registra todo mediante logging.

In [72]:
def download_and_clean(url, output_name, sep=',', date_columns=None, numeric_columns=None):
    """
    Descarga un CSV, limpia tipos básicos y lo guarda.
    
    Parámetros:
    - url: URL completa del archivo CSV.
    - output_name: nombre base para el archivo guardado (sin extensión).
    - sep: separador de columnas (; o ,).
    - date_columns: lista de columnas que deben convertirse a datetime.
    - numeric_columns: lista de columnas que tienen comas como decimal y deben ser numéricas.
    
    Retorna:
    - DataFrame limpio o None si hay error.
    """
    try:
        logger.info(f"📥 Descargando {output_name} desde {url}")
        
        # Descargar el CSV
        df = pd.read_csv(
            url,
            sep=sep,
            encoding='utf-8',
            on_bad_lines='skip',   # Salta líneas mal formadas (robusto)
            engine='python'         # Más tolerante con variaciones
        )
        logger.info(f"   → {len(df)} filas descargadas")
        
        # 1. Convertir columnas de fecha
        if date_columns:
            for col in date_columns:
                if col in df.columns:
                    antes_nulos = df[col].isna().sum()
                    df[col] = pd.to_datetime(df[col], errors='coerce')
                    despues_nulos = df[col].isna().sum()
                    if despues_nulos > antes_nulos:
                        logger.warning(f"   ⚠️ {col}: {despues_nulos - antes_nulos} fechas inválidas convertidas a NaT")
                else:
                    logger.warning(f"   ⚠️ Columna fecha {col} no encontrada en {output_name}")
        
        # 2. Limpiar columnas numéricas que vienen con coma decimal
        if numeric_columns:
            for col in numeric_columns:
                if col in df.columns:
                    # Convertir a string, reemplazar coma por punto, luego a número
                    df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                    nulos_nuevos = df[col].isna().sum()
                    if nulos_nuevos > 0:
                        logger.warning(f"   ⚠️ {col}: {nulos_nuevos} valores no numéricos convertidos a NaN")
                else:
                    logger.warning(f"   ⚠️ Columna numérica {col} no encontrada en {output_name}")
        
        # 3. Guardar el DataFrame limpio
        output_path = DATA_LOADED / f"{output_name}_raw.csv"
        df.to_csv(output_path, index=False, encoding='utf-8')
        logger.info(f"   💾 Guardado en {output_path} ({len(df)} filas, {len(df.columns)} columnas)")
        
        return df
        
    except Exception as e:
        logger.error(f"❌ Error procesando {output_name}: {e}")
        return None

### <font color='lightgreen'>Ejecutar descarga para todos los archivos</font>

Iteramos sobre la configuración y llamamos a la función genérica.
Los DataFrames se almacenan en un diccionario para posible uso posterior en el notebook.

In [73]:
datos = {}   # Diccionario para guardar los DataFrames si los necesitamos más adelante

for filename, name, sep, date_cols, num_cols in ARCHIVOS_CONFIG:
    url = f"{BASE_URL}/{filename}"
    df = download_and_clean(url, name, sep=sep, date_columns=date_cols, numeric_columns=num_cols)
    if df is not None:
        datos[name] = df

2026-04-15 03:36:39,123 - INFO - 📥 Descargando compras desde https://raw.githubusercontent.com/No-Country-simulation/S03-26-Equipo-47-Business-Intelligence/development/data/raw/compras_proveedor.csv
2026-04-15 03:36:39,821 - INFO -    → 864 filas descargadas
2026-04-15 03:36:39,933 - INFO -    💾 Guardado en c:\Users\marely\OneDrive\Documentos\Bussiness Inteligence Project\S03-26-Equipo-47-Business-Intelligence-development\S03-26-Equipo-47-Business-Intelligence-development\data\loaded\compras_raw.csv (864 filas, 8 columnas)
2026-04-15 03:36:40,177 - INFO - 📥 Descargando consumo_energetico desde https://raw.githubusercontent.com/No-Country-simulation/S03-26-Equipo-47-Business-Intelligence/development/data/raw/consumo_energetico.csv
2026-04-15 03:36:40,533 - INFO -    → 178 filas descargadas
2026-04-15 03:36:40,547 - INFO -    💾 Guardado en c:\Users\marely\OneDrive\Documentos\Bussiness Inteligence Project\S03-26-Equipo-47-Business-Intelligence-development\S03-26-Equipo-47-Business-Intelli

### <font color='lightgreen'>Verificación de integridad</font>

Comprobamos que se hayan generado todos los archivos esperados.

In [74]:
archivos_esperados = [f"{name}_raw.csv" for _, name, _, _, _ in ARCHIVOS_CONFIG]
archivos_generados = list(DATA_LOADED.glob("*.csv"))

logger.info("\n" + "="*50)
logger.info("VERIFICACIÓN DE ARCHIVOS GENERADOS")
logger.info("="*50)

for esperado in archivos_esperados:
    if (DATA_LOADED / esperado).exists():
        logger.info(f"✅ {esperado}")
    else:
        logger.error(f"❌ FALTA {esperado}")

2026-04-15 03:36:48,960 - INFO - 
2026-04-15 03:36:48,962 - INFO - VERIFICACIÓN DE ARCHIVOS GENERADOS
2026-04-15 03:36:48,964 - INFO - ==================================================
2026-04-15 03:36:48,966 - INFO - ✅ compras_raw.csv
2026-04-15 03:36:48,968 - INFO - ✅ consumo_energetico_raw.csv
2026-04-15 03:36:48,971 - INFO - ✅ encuestas_raw.csv
2026-04-15 03:36:48,975 - INFO - ✅ eventos_rrhh_raw.csv
2026-04-15 03:36:48,981 - INFO - ✅ personal_nomina_raw.csv
2026-04-15 03:36:48,985 - INFO - ✅ residuos_raw.csv
2026-04-15 03:36:48,988 - INFO - ✅ ventas_raw.csv


### <font color='lightgreen'>Resumen final</font>

In [75]:
print("\n" + "="*50)
print("✅ CARGA COMPLETADA")
print("="*50)
print(f"\n📊 Dataframes cargados en memoria:")
for name, df in datos.items():
    print(f"   • {name}: {len(df):,} filas, {len(df.columns)} columnas")

print(f"\n📁 Archivos guardados en: {DATA_LOADED}/")
print("\n🔍 Siguiente paso: Ejecutar 02_clean_data.ipynb")


✅ CARGA COMPLETADA

📊 Dataframes cargados en memoria:
   • compras: 864 filas, 8 columnas
   • consumo_energetico: 178 filas, 12 columnas
   • encuestas: 189 filas, 12 columnas
   • eventos_rrhh: 537 filas, 10 columnas
   • personal_nomina: 1,378 filas, 12 columnas
   • residuos: 2,508 filas, 10 columnas
   • ventas: 43,893 filas, 15 columnas

📁 Archivos guardados en: c:\Users\marely\OneDrive\Documentos\Bussiness Inteligence Project\S03-26-Equipo-47-Business-Intelligence-development\S03-26-Equipo-47-Business-Intelligence-development\data\loaded/

🔍 Siguiente paso: Ejecutar 02_clean_data.ipynb
